# YOLOv8s 训练 - Block 检测

在 **百度 AI Studio** 上训练自定义 YOLO 模型。

## 前提
1. 注册 AI Studio：https://aistudio.baidu.com
2. 右侧 **运行环境** 选择 **V100（16GB）** 免费版
3. 先用 `prepare_dataset.py` 处理好数据集 → 上传 `dataset_switch.zip` 到 AI Studio 数据集

按顺序运行每个单元格即可。

In [ ]:
# 1. 安装依赖
!pip install ultralytics -q
from ultralytics import YOLO
import os

In [ ]:
# 2. 从 AI Studio 数据集加载
# 数据集会挂载在 /home/aistudio/data/ 下

import zipfile

# 找到上传的数据集 zip
data_dir = '/home/aistudio/data'
zip_files = [f for f in os.listdir(data_dir) if f.endswith('.zip')]
print(f'找到数据集: {zip_files}')

if zip_files:
    zip_path = os.path.join(data_dir, zip_files[0])
    print(f'解压: {zip_path}')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall('/home/aistudio/dataset')
    print('✅ 解压完成')
else:
    print('❌ 未找到 zip 文件，请检查数据集上传')
    !ls -la {data_dir}

# 检查
if os.path.exists('/home/aistudio/dataset/data.yaml'):
    train_count = len(os.listdir('/home/aistudio/dataset/images/train'))
    val_count = len(os.listdir('/home/aistudio/dataset/images/val'))
    print(f'📊 训练集: {train_count} 张')
    print(f'📊 验证集: {val_count} 张')
    print('✅ 数据集准备就绪！')
else:
    print('❌ data.yaml 未找到')
    !ls -la /home/aistudio/dataset/

In [ ]:
# 3. 查看数据集配置
!cat /home/aistudio/dataset/data.yaml

In [ ]:
# 4. 开始训练 YOLOv8s（block 检测）
import torch
print(f'🚀 GPU 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU 型号: {torch.cuda.get_device_name(0)}')

model = YOLO('yolov8s.pt')

model.train(
    data='/home/aistudio/dataset/data.yaml',
    epochs=200,                # 训练轮数
    imgsz=640,                 # 输入图片尺寸
    batch=16,                  # 批次大小（V100 16GB）
    device='cuda',
    workers=2,
    patience=50,               # 50轮无提升就早停
    name='custom_block',
    exist_ok=True,
    verbose=True,
    plots=True,                # 生成训练曲线
    save=True,
)

In [ ]:
# 5. 查看训练结果
print('📁 权重文件:')
!ls -lh /home/aistudio/dataset/runs/detect/custom_block/weights/
print()
print('📊 训练曲线:')
!ls /home/aistudio/dataset/runs/detect/custom_block/*.png

In [ ]:
# 6. 下载 best.pt 到本地
# AI Studio 左侧文件列表可以直接下载
# 或者用下面代码压缩后下载

best_pt = '/home/aistudio/dataset/runs/detect/custom_block/weights/best.pt'

if os.path.exists(best_pt):
    import shutil
    shutil.copy(best_pt, '/home/aistudio/best.pt')
    print('✅ best.pt 已复制到 /home/aistudio/best.pt')
    print('   → 在左侧文件列表中右键下载')
else:
    print('❌ best.pt 未找到')

print()
print('📌 下载后放到本地:')
print('   src/tl_vision/model/best.pt')
print('   并修改 config/yolo_node.yaml 中的 model_path')

---

## ⚡ 断点续训

如果训练被中断（关浏览器、断网等），重新打开 Notebook 后：
1. 重新运行 ① ② ③ 单元格（安装 + 解压）
2. 运行下面单元格从中断处继续

In [ ]:
# [断点续训] 从上次中断处继续
last_ckpt = '/home/aistudio/dataset/runs/detect/custom_block/weights/last.pt'

if os.path.exists(last_ckpt):
    print(f'✅ 找到断点: {last_ckpt}')
    model = YOLO(last_ckpt)
    model.train(resume=True)
else:
    print('❌ 未找到断点，请先完整跑一次训练')